In [1]:
import os

In [2]:
import pandas as pd
import numpy as np
import torch
import torchvision
from torchvision.transforms import v2 as T
from pathlib import Path
import matplotlib.pyplot as plt

In [3]:
from torch.utils.data import DataLoader, Subset, Dataset

In [4]:
from PIL import Image

In [43]:
attributes=pd.read_csv("/kaggle/input/datasets/adityajigyasu/celebrity-dataset/celeba/list_attr_celeba.txt")

In [44]:
attributes.to_csv('attributes.csv',index=None)

In [7]:
count=0
for r in attributes.iloc[0].to_string().split():
    print(r)
    print()

202599

5_o_Clock_Shadow

Arched_Eyebrows

Attractive

Ba...



In [8]:
count=0
for i in attributes.iloc[0].to_list()[0].split():
    print(i)
    count=count+1
print(count)

5_o_Clock_Shadow
Arched_Eyebrows
Attractive
Bags_Under_Eyes
Bald
Bangs
Big_Lips
Big_Nose
Black_Hair
Blond_Hair
Blurry
Brown_Hair
Bushy_Eyebrows
Chubby
Double_Chin
Eyeglasses
Goatee
Gray_Hair
Heavy_Makeup
High_Cheekbones
Male
Mouth_Slightly_Open
Mustache
Narrow_Eyes
No_Beard
Oval_Face
Pale_Skin
Pointy_Nose
Receding_Hairline
Rosy_Cheeks
Sideburns
Smiling
Straight_Hair
Wavy_Hair
Wearing_Earrings
Wearing_Hat
Wearing_Lipstick
Wearing_Necklace
Wearing_Necktie
Young
40


In [9]:
print(attributes.iloc[0].to_string())

202599    5_o_Clock_Shadow Arched_Eyebrows Attractive Ba...


In [45]:
features=attributes.iloc[0].to_list()[0].split()


In [47]:
column_name=[]
for i,_ in enumerate(features):
    column_name.append(i)

In [49]:
attributes=attributes.iloc[1:,0].str.split(expand=True)
attributes.columns=['image_file']+column_name


In [50]:
attributes

,image_file,0,1,2,3,4,5,6,7,8,...,30,31,32,33,34,35,36,37,38,39
1,000001.jpg,-1,1,1,-1,-1,-1,-1,-1,-1,...,-1,1,1,-1,1,-1,1,-1,-1,1
2,000002.jpg,-1,-1,-1,1,-1,-1,-1,1,-1,...,-1,1,-1,-1,-1,-1,-1,-1,-1,1
3,000003.jpg,-1,-1,-1,-1,-1,-1,1,-1,-1,...,-1,-1,-1,1,-1,-1,-1,-1,-1,1
4,000004.jpg,-1,-1,1,-1,-1,-1,-1,-1,-1,...,-1,-1,1,-1,1,-1,1,1,-1,1
5,000005.jpg,-1,1,1,-1,-1,-1,1,-1,-1,...,-1,-1,-1,-1,-1,-1,1,-1,-1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
202595,202595.jpg,-1,-1,1,-1,-1,-1,1,-1,-1,...,-1,-1,-1,-1,-1,-1,1,-1,-1,1
202596,202596.jpg,-1,-1,-1,-1,-1,1,1,-1,-1,...,-1,1,1,-1,-1,-1,-1,-1,-1,1
202597,202597.jpg,-1,-1,-1,-1,-1,-1,-1,-1,1,...,-1,1,-1,-1,-1,-1,-1,-1,-1,1
202598,202598.jpg,-1,1,1,-1,-1,-1,1,-1,1,...,-1,1,-1,1,1,-1,1,-1,-1,1


In [51]:
attributes[column_name]=attributes[column_name].astype(int)

In [52]:
attributes

,image_file,0,1,2,3,4,5,6,7,8,...,30,31,32,33,34,35,36,37,38,39
1,000001.jpg,-1,1,1,-1,-1,-1,-1,-1,-1,...,-1,1,1,-1,1,-1,1,-1,-1,1
2,000002.jpg,-1,-1,-1,1,-1,-1,-1,1,-1,...,-1,1,-1,-1,-1,-1,-1,-1,-1,1
3,000003.jpg,-1,-1,-1,-1,-1,-1,1,-1,-1,...,-1,-1,-1,1,-1,-1,-1,-1,-1,1
4,000004.jpg,-1,-1,1,-1,-1,-1,-1,-1,-1,...,-1,-1,1,-1,1,-1,1,1,-1,1
5,000005.jpg,-1,1,1,-1,-1,-1,1,-1,-1,...,-1,-1,-1,-1,-1,-1,1,-1,-1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
202595,202595.jpg,-1,-1,1,-1,-1,-1,1,-1,-1,...,-1,-1,-1,-1,-1,-1,1,-1,-1,1
202596,202596.jpg,-1,-1,-1,-1,-1,1,1,-1,-1,...,-1,1,1,-1,-1,-1,-1,-1,-1,1
202597,202597.jpg,-1,-1,-1,-1,-1,-1,-1,-1,1,...,-1,1,-1,-1,-1,-1,-1,-1,-1,1
202598,202598.jpg,-1,1,1,-1,-1,-1,1,-1,1,...,-1,1,-1,1,1,-1,1,-1,-1,1


In [64]:
features[31]

'Smiling'

In [65]:
smiling_attr=attributes[31].to_list()

In [15]:
image_repo=Path('/kaggle/input/datasets/adityajigyasu/celebrity-dataset/celeba/img_align_celeba/img_align_celeba')

In [16]:
image_repo=[image for image in image_repo.iterdir()]

In [17]:
len(image_repo)

202599

In [77]:
class CelebDataset(Dataset):
    
    def __init__(self,image,labels):
        super().__init__()
        self.image=image
        self.labels=labels
    def __len__(self):
        return len(self.image)
    def __getitem__(self,idx):
        img=self.image[idx]
        label=self.labels[idx]
        return img,label

In [78]:
import pandas as pd

data={0:[1,2,3],1:[2,3,4]}
df=pd.DataFrame(data,index=['A','B','C'])

In [79]:
df

,0,1
A,1,2
B,2,3
C,3,4


In [80]:
dataset=CelebDataset(image_repo,smiling_attr)

In [94]:
import random

In [95]:
smile=[]
non_smile=[]
for i,label in enumerate(smiling_attr):
    if label==1:
      smile.append(i)
    else:
      non_smile.append(i)

In [97]:
smile_sample=random.sample(smile,len(smile))
non_smile_sample=random.sample(non_smile,len(non_smile))

In [104]:
print("Same size divide:",len(smile)+len(non_smile)==len(smiling_attr))

Same size divide: True


In [113]:
len_train_smile=int(0.7*len(smile_sample))
len_val_smile=len_train_smile+int(0.15*len(smile_sample))

train_smile_idx=smile_sample[:len_train_smile]
val_smile_idx=smile_sample[len_train_smile:len_val_smile]
test_smile_idx=smile_sample[len_val_smile:]

In [114]:
len_train_non_smile=int(0.7*len(non_smile_sample))
len_val_non_smile=len_train_non_smile+int(0.15*len(non_smile_sample))

train_non_smile_idx=non_smile_sample[:len_train_non_smile]
val_non_smile_idx=non_smile_sample[len_train_non_smile:len_val_non_smile]
test_non_smile_idx=non_smile_sample[len_val_non_smile:]

In [123]:
train_idx=train_smile_idx+train_non_smile_idx
test_idx=test_smile_idx+test_non_smile_idx
val_idx=val_smile_idx+val_non_smile_idx

train_idx=random.sample(train_idx,len(train_idx))
test_idx=random.sample(test_idx,len(test_idx))
val_idx=random.sample(val_idx,len(val_idx))

In [124]:
train_dataset=Subset(dataset,train_idx)
val_dataset=Subset(dataset,val_idx)
test_dataset=Subset(dataset,test_idx)